# Attention Mechanism:
1. The Encoder will use Bidirectional LSTM.
2. The Decoder gets values of the fix number of inputs after going through ENCODER:
     1. with soft attention → window = whole sentence.
     2. local/sliding attention → window = only a fixed number of nearby tokens.(These fixed numbers are called Window Size)
3. Alpha will be initialize with every input/vectors goes to the Decoder.(Just like in ANN)
4. All Alpha values will be equal to 1.(Because of Softmax)
5. This Alpha Values gets changes in BackPropogation.

In [1]:
import pandas as pd

dataset=pd.read_json('/home/hammadali08/Personal/FYP Datasets/News_Category_Dataset_v3.json',lines=True)
df=dataset.head(700)
df

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22
...,...,...,...,...,...,...
695,https://www.huffpost.com/entry/new-jersey-expa...,New Jersey Governor Announces Proposals To Exp...,POLITICS,"“Without access, rights mean nothing,” Gov. Ph...",Ryan Grenoble,2022-05-11
696,https://www.huffpost.com/entry/cori-bush-abort...,Rep. Cori Bush: Biden Should 'Absolutely' Say ...,POLITICS,The Missouri Democratic congresswoman has shar...,Amanda Terkel,2022-05-11
697,https://www.huffpost.com/entry/bc-us-virus-out...,COVID-19 Cases Rise While Few School Mask Mand...,U.S. NEWS,"U.S. coronavirus cases are up, leading a smatt...","STEVE LeBLANC and MIKE CATALINI, AP",2022-05-11
698,https://www.huffpost.com/entry/engagement-ring...,Man Pops The Question After Engagement Ring Su...,WEIRD NEWS,Myers Hart said he took the incident as a “sig...,Jazmin Tolliver,2022-05-11


# Input Preprocessing

In [2]:
import re
corpus = []
for i in range(0, len(df)):
    review = re.sub('[^a-zA-Z]', ' ', df['headline'][i])
    review = review.lower()
    review = review.split()

    review = ' '.join(review)
    corpus.append(review)

In [3]:
corpus

['over million americans roll up sleeves for omicron targeted covid boosters',
 'american airlines flyer charged banned for life after punching flight attendant on video',
 'of the funniest tweets about cats and dogs this week sept',
 'the funniest tweets from parents this week sept',
 'woman who called cops on black bird watcher loses lawsuit against ex employer',
 'cleaner was dead in belk bathroom for days before body found police',
 'reporter gets adorable surprise from her boyfriend while live on tv',
 'puerto ricans desperate for water after hurricane fiona s rampage',
 'how a new documentary captures the complexity of being a child of immigrants',
 'biden at un to call russian war an affront to body s charter',
 'world cup captains want to wear rainbow armbands in qatar',
 'man sets himself on fire in apparent protest of funeral for japan s abe',
 'fiona threatens to become category storm headed to bermuda',
 'twitch bans gambling sites after streamer scams folks out of',
 'virg

In [4]:
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess
words=[]
for sent in corpus:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))   # Simple Preprcoess make it lowe r case
words

[['over',
  'million',
  'americans',
  'roll',
  'up',
  'sleeves',
  'for',
  'omicron',
  'targeted',
  'covid',
  'boosters'],
 ['american',
  'airlines',
  'flyer',
  'charged',
  'banned',
  'for',
  'life',
  'after',
  'punching',
  'flight',
  'attendant',
  'on',
  'video'],
 ['of',
  'the',
  'funniest',
  'tweets',
  'about',
  'cats',
  'and',
  'dogs',
  'this',
  'week',
  'sept'],
 ['the', 'funniest', 'tweets', 'from', 'parents', 'this', 'week', 'sept'],
 ['woman',
  'who',
  'called',
  'cops',
  'on',
  'black',
  'bird',
  'watcher',
  'loses',
  'lawsuit',
  'against',
  'ex',
  'employer'],
 ['cleaner',
  'was',
  'dead',
  'in',
  'belk',
  'bathroom',
  'for',
  'days',
  'before',
  'body',
  'found',
  'police'],
 ['reporter',
  'gets',
  'adorable',
  'surprise',
  'from',
  'her',
  'boyfriend',
  'while',
  'live',
  'on',
  'tv'],
 ['puerto',
  'ricans',
  'desperate',
  'for',
  'water',
  'after',
  'hurricane',
  'fiona',
  'rampage'],
 ['how',
  'new',


In [5]:
all_words = [word for sentence in words for word in sentence]

# Get unique words
unique_words_input = set(all_words)

# Count them
print("Number of unique words:", len(unique_words_input))
print("Unique words:", unique_words_input)

Number of unique words: 3217
Unique words: {'minority', 'thousands', 'shirt', 'cameron', 'might', 'hit', 're', 'disguising', 'alligators', 'abortions', 'algae', 'ban', 'tour', 'computer', 'louisville', 'sweltering', 'prescription', 'theme', 'shoplifting', 'jungle', 'armas', 'holly', 'they', 'avert', 'republican', 'dr', 'cosmologists', 'woody', 'liv', 'turkey', 'daniels', 'starbucks', 'complains', 'streamer', 'arrest', 'have', 'rainbow', 'six', 'emergency', 'dna', 'atop', 'rolling', 'slams', 'giuliani', 'profound', 'pops', 'gas', 'hears', 'mall', 'eyed', 'cow', 'glaser', 'glimpse', 'reconcile', 'hawaii', 'post', 'happy', 'characters', 'utah', 'run', 'highest', 'takeaways', 'saving', 'professor', 'weinstein', 'grant', 'hike', 'fined', 'no', 'airport', 'sidekick', 'depp', 'equality', 'panel', 'soon', 'sidesteps', 'detained', 'firebomb', 'engagement', 'health', 'alexandria', 'campaigning', 'djokovic', 'gig', 'knew', 'donates', 'griner', 'accuses', 'bleeped', 'killed', 'news', 'nasty', 'ani

In [6]:
max_len_input = max(len(sentence) for sentence in words)
longest_sentence = max(words, key=len)

print("Max number of words:", max_len_input)
print("Sentence with max words:", longest_sentence)

Max number of words: 17
Sentence with max words: ['alex', 'jones', 'sandy', 'hook', 'defamation', 'trial', 'is', 'set', 'to', 'begin', 'here', 'how', 'it', 'got', 'to', 'this', 'point']


In [7]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

2025-09-02 22:27:26.185652: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-02 22:27:26.336385: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-02 22:27:26.388070: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756834046.514361   24973 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756834046.543190   24973 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756834046.744293   24973 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [8]:
one_hot_input=[one_hot(word, len(unique_words_input)) for word in corpus]
one_hot_input

[[2974, 1084, 1396, 710, 767, 3151, 20, 2348, 1866, 45, 2348],
 [2578, 674, 3057, 2125, 1882, 20, 1539, 983, 44, 1544, 228, 990, 2679],
 [390, 1701, 2327, 161, 161, 2769, 2534, 1899, 2218, 1367, 2943],
 [1701, 2327, 161, 1683, 1097, 2218, 1367, 2943],
 [463, 2014, 360, 822, 990, 1161, 1344, 968, 776, 2323, 2563, 669, 2849],
 [91, 469, 3207, 1535, 371, 1169, 20, 22, 984, 2355, 2928, 267],
 [3036, 320, 2785, 666, 1683, 33, 56, 1853, 1008, 990, 3067],
 [1890, 1123, 2532, 20, 2786, 983, 1855, 1429, 309, 1933],
 [959, 2473, 2065, 2229, 2377, 1701, 39, 390, 2988, 2473, 386, 390, 3034],
 [952, 2770, 2952, 1091, 2372, 136, 2680, 2978, 434, 1091, 2355, 309, 2071],
 [2601, 1261, 1848, 3201, 1091, 907, 2385, 2936, 1535, 563],
 [1218, 791, 742, 990, 2611, 1535, 1096, 1947, 390, 2750, 20, 99, 309, 516],
 [1429, 2491, 1091, 1877, 1712, 2191, 1526, 1091, 2846],
 [114, 1182, 3047, 1318, 983, 2905, 2117, 1260, 3001, 390],
 [2438, 1237, 347, 1091, 36, 2865, 696, 288],
 [136,
  1149,
  306,
  287,
  2014

In [9]:
input_padding=pad_sequences(one_hot_input, maxlen=max_len_input, padding='post')
input_padding

array([[2974, 1084, 1396, ...,    0,    0,    0],
       [2578,  674, 3057, ...,    0,    0,    0],
       [ 390, 1701, 2327, ...,    0,    0,    0],
       ...,
       [  45,  526,  873, ...,    0,    0,    0],
       [1218, 1975, 1701, ...,    0,    0,    0],
       [2607,  581, 1505, ...,    0,    0,    0]], dtype=int32)

# Output Presprocessing

In [10]:
corpus1 = []
for i in range(0, len(df)):
    review = re.sub('[^a-zA-Z]', ' ', df['short_description'][i])
    review = review.lower()
    review = review.split()

    review = ' '.join(review)
    corpus1.append(review)

In [11]:
corpus1

['health experts said it is too early to predict whether demand would match up with the million doses of the new boosters the u s ordered for the fall',
 'he was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation according to the u s attorney s office in los angeles',
 'until you have a dog you don t understand what could be eaten',
 'accidentally put grown up toothpaste on my toddler s toothbrush and he screamed like i was cleaning his teeth with a carolina reaper dipped in tabasco sauce',
 'amy cooper accused investment firm franklin templeton of unfairly firing her and branding her a racist after video of the central park encounter went viral',
 'the year old woman was seen working at the south carolina store on thursday she was found dead monday after her family reported her missing authorities said',
 'who s that behind you an anchor for new york s pix asked journalist michelle ross as she finished up an interview',
 'more than half a m

In [12]:
decoder_input_data = ["<start> " + txt + "" for txt in corpus1]
decoder_target_data= ["" + txt + " <end>" for txt in corpus1]
decoder_input_data

['<start> health experts said it is too early to predict whether demand would match up with the million doses of the new boosters the u s ordered for the fall',
 '<start> he was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation according to the u s attorney s office in los angeles',
 '<start> until you have a dog you don t understand what could be eaten',
 '<start> accidentally put grown up toothpaste on my toddler s toothbrush and he screamed like i was cleaning his teeth with a carolina reaper dipped in tabasco sauce',
 '<start> amy cooper accused investment firm franklin templeton of unfairly firing her and branding her a racist after video of the central park encounter went viral',
 '<start> the year old woman was seen working at the south carolina store on thursday she was found dead monday after her family reported her missing authorities said',
 '<start> who s that behind you an anchor for new york s pix asked journalist michelle ros

In [13]:
words1=[]
for sent in corpus1:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words1.append(simple_preprocess(sent))
words1

[['health',
  'experts',
  'said',
  'it',
  'is',
  'too',
  'early',
  'to',
  'predict',
  'whether',
  'demand',
  'would',
  'match',
  'up',
  'with',
  'the',
  'million',
  'doses',
  'of',
  'the',
  'new',
  'boosters',
  'the',
  'ordered',
  'for',
  'the',
  'fall'],
 ['he',
  'was',
  'subdued',
  'by',
  'passengers',
  'and',
  'crew',
  'when',
  'he',
  'fled',
  'to',
  'the',
  'back',
  'of',
  'the',
  'aircraft',
  'after',
  'the',
  'confrontation',
  'according',
  'to',
  'the',
  'attorney',
  'office',
  'in',
  'los',
  'angeles'],
 ['until',
  'you',
  'have',
  'dog',
  'you',
  'don',
  'understand',
  'what',
  'could',
  'be',
  'eaten'],
 ['accidentally',
  'put',
  'grown',
  'up',
  'toothpaste',
  'on',
  'my',
  'toddler',
  'toothbrush',
  'and',
  'he',
  'screamed',
  'like',
  'was',
  'cleaning',
  'his',
  'teeth',
  'with',
  'carolina',
  'reaper',
  'dipped',
  'in',
  'tabasco',
  'sauce'],
 ['amy',
  'cooper',
  'accused',
  'investmen

In [14]:
max_len_output = max(len(sentence) for sentence in words1)
longest_sentence = max(words1, key=len)

print("Sentence with max words:", longest_sentence)
max_len_output= max_len_output+1            # As there will be <start> and <end>
print("Max number of words:", max_len_output)

Sentence with max words: ['when', 'elsa', 'avila', 'looks', 'at', 'the', 'scar', 'that', 'runs', 'down', 'her', 'torso', 'she', 'can', 'help', 'remember', 'may', 'when', 'gunman', 'stormed', 'her', 'fourth', 'grade', 'wing', 'at', 'robb', 'elementary', 'in', 'uvalde', 'texas', 'killing', 'children', 'and', 'two', 'teachers', 'and', 'leaving', 'her', 'and', 'others', 'wounded']
Max number of words: 42


In [15]:
all_words1 = [word for sentence in words1 for word in sentence]

unique_words_output = set(all_words1)

# Count them
print("Number of unique words:", len(unique_words_output))
print("Unique words:", unique_words_output)

Number of unique words: 4202
Unique words: {'minority', 'thousands', 'baird', 'kathleen', 'moore', 'shirt', 'might', 'folk', 'hit', 'macomb', 're', 'uncertain', 'hasn', 'proposed', 'services', 'vaccination', 'pact', 'atlanta', 'abortions', 'geology', 'testified', 'ban', 'religious', 'tour', 'lookalike', 'louisville', 'triplets', 'dreams', 'zaslav', 'ownership', 'activists', 'jungle', 'armas', 'annual', 'they', 'joel', 'republican', 'dr', 'dropping', 'athletes', 'woody', 'liv', 'turkey', 'daniels', 'starbucks', 'gorske', 'continuing', 'arrest', 'have', 'missed', 'russell', 'decimated', 'happens', 'examines', 'legitimate', 'teenager', 'six', 'emergency', 'treatments', 'finished', 'welcome', 'poolside', 'likes', 'rolling', 'lob', 'assailant', 'gas', 'ross', 'price', 'mall', 'laden', 'lucy', 'newest', 'consensual', 'receiving', 'post', 'requirements', 'griped', 'happy', 'characters', 'unlike', 'utah', 'influenced', 'run', 'blood', 'highest', 'crazy', 'tweeted', 'moricz', 'resettling', 'mir

In [16]:
decoder_input = [one_hot(word, len(unique_words_output)) for word in decoder_input_data]
decoder_target = [one_hot(word, len(unique_words_output)) for word in decoder_target_data]
decoder_target

[[1130,
  913,
  837,
  2187,
  1959,
  3498,
  2001,
  1979,
  1887,
  2274,
  1956,
  2728,
  3595,
  3166,
  1670,
  261,
  2612,
  2139,
  2879,
  261,
  175,
  773,
  261,
  2770,
  472,
  998,
  224,
  261,
  1906,
  2847],
 [4174,
  4052,
  254,
  3470,
  1944,
  2338,
  3118,
  3557,
  4174,
  1916,
  1979,
  261,
  3693,
  2879,
  261,
  3461,
  3537,
  261,
  2550,
  1227,
  1979,
  261,
  2770,
  472,
  1025,
  472,
  1440,
  195,
  1213,
  418,
  2847],
 [500,
  3089,
  1409,
  1835,
  620,
  3089,
  2066,
  898,
  1523,
  3297,
  375,
  1111,
  886,
  2847],
 [235,
  3980,
  2281,
  3166,
  1846,
  1501,
  3644,
  562,
  472,
  1312,
  2338,
  4174,
  70,
  2152,
  2415,
  4052,
  2171,
  1699,
  2167,
  1670,
  1835,
  1856,
  3084,
  1745,
  195,
  679,
  2688,
  2847],
 [1528,
  3026,
  3509,
  1669,
  1052,
  3240,
  3951,
  2879,
  1724,
  543,
  2577,
  2338,
  2768,
  2577,
  1835,
  2503,
  3537,
  71,
  2879,
  261,
  3266,
  3796,
  3884,
  3897,
  3699,
  2847],

In [17]:
decoder_input_data=pad_sequences(decoder_input, maxlen=max_len_output, padding='post')
decoder_input_data.shape

(700, 42)

In [18]:
decoder_target_data=pad_sequences(decoder_target, maxlen=max_len_output, padding='post')
decoder_target_data.shape

(700, 42)

In [19]:
from tensorflow.keras.utils import to_categorical

decoder_target_data = to_categorical(
    decoder_target_data,
    num_classes=len(unique_words_output)     # same as your output layer vocab size
)
decoder_target_data.shape

(700, 42, 4202)

# Attention model:

In [20]:
from tensorflow.keras.layers import Bidirectional, Attention, Concatenate, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Attention, Concatenate

# ===== Encoder with Bidirectional LSTM =====
encoder_inputs = Input(shape=(None,))
x = Embedding(input_dim=len(unique_words_input), output_dim=128, input_length=max_len_input)(encoder_inputs)
# Use Bidirectional LSTM
encoder_lstm = Bidirectional(LSTM(256, return_sequences=True, return_state=True))
encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(x)

# Concatenate the forward and backward states
state_h = Concatenate()([forward_h, backward_h])
state_c = Concatenate()([forward_c, backward_c])

# Encoder states (to pass into decoder)
encoder_states = [state_h, state_c]

# ===== Decoder with Attention =====
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(input_dim=len(unique_words_output), output_dim=128, input_length=max_len_output)(decoder_inputs)

# LSTM decoder, initialized with encoder states
decoder_lstm = LSTM(512, return_sequences=True, return_state=True) # Decoder LSTM size should match combined encoder states
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# Attention layer
attention = Attention()([decoder_outputs, encoder_outputs])

# Concatenate attention output with decoder outputs
decoder_concat_input = Concatenate(axis=-1)([decoder_outputs, attention])

# Final dense layer to predict word probabilities
decoder_dense = Dense(len(unique_words_output), activation='softmax')
decoder_outputs = decoder_dense(decoder_concat_input)


# Define the model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

/home/hammadali08/.local/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-09-02 22:27:32.591960: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │    411,776 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ [(None, None,     │    788,480 │ embedding[0][0]   │
│ (Bidirectional)     │ 512), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │    537,856 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │  1,312,768 │ embedding_1[0][0… │
│                     │ 512), (None,      │            │ concatenate[0][0… │
│                     │ 512), (None,      │            │ concatenate_1[0]… │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 512) │          0 │ lstm_1[0][0],     │
│ (Attention)         │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, None,      │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │ 1024)             │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  4,307,050 │ concatenate_2[0]… │
│                     │ 4202)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,357,930 (28.07 MB)

 Trainable params: 7,357,930 (28.07 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.fit([input_padding, decoder_input_data], decoder_target_data,
          batch_size=50,
          epochs=3,validation_split=0.2)

Epoch 1/3


2025-09-02 22:27:34.087015: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 395324160 exceeds 10% of free system memory.
2025-09-02 22:27:38.335911: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 860569600 exceeds 10% of free system memory.


 1/12 ━━━━━━━━━━━━━━━━━━━━ 59s 5s/step - accuracy: 4.7619e-04 - loss: 8.3432

2025-09-02 22:27:40.243186: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 860569600 exceeds 10% of free system memory.


 2/12 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.1224 - loss: 8.3269    

2025-09-02 22:27:42.165010: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 860569600 exceeds 10% of free system memory.


 3/12 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.1959 - loss: 8.3013

2025-09-02 22:27:43.622619: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 860569600 exceeds 10% of free system memory.


12/12 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.4470 - loss: 6.1342 - val_accuracy: 0.4968 - val_loss: 4.1291
Epoch 2/3
12/12 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.4902 - loss: 3.9806 - val_accuracy: 0.5031 - val_loss: 3.8902
Epoch 3/3
12/12 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5001 - loss: 3.6295 - val_accuracy: 0.5107 - val_loss: 3.5931
